In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import pickle

# 1. Load the Master Data
print("Loading Master Dataset (43,812 rows)...")
df = pd.read_csv("global_weather_master.csv")
df.head()

Loading Master Dataset (43,812 rows)...


,City_Type,Date,Temp_Max_C,Temp_Mean_C,Humidity_Mean_pct,Wind_Speed_max_kmh,Precipitation_mm
0,Delhi_India,2016-05-04,38.9,30.9,48,20.6,1.0
1,Delhi_India,2016-05-05,34.6,29.4,50,20.6,0.1
2,Delhi_India,2016-05-06,34.1,29.2,54,9.4,0.0
3,Delhi_India,2016-05-07,36.0,30.8,41,11.4,0.0
4,Delhi_India,2016-05-08,37.2,30.9,45,14.6,0.0


In [7]:
# 2. Advanced Feature Engineering
print("🛠️ Engineering Pro Features...")
# Fluctuation capture karta hai sudden weather changes
df['Temp_Fluctuation'] = df['Temp_Max_C'] - df['Temp_Mean_C']

# Heat-Humidity Index (Feel-like Proxy)
df['Heat_Humidity_Index'] = df['Temp_Mean_C'] * (df['Humidity_Mean_pct'] / 100.0)

# Binary Rain State
df['Is_Raining'] = (df['Precipitation_mm'] > 0).astype(int)

🛠️ Engineering Pro Features...


In [8]:
# 3. Target Label Generation (Rule-Based Expert System)
def calculate_health_risk(row):
    temp = row['Temp_Mean_C']
    humidity = row['Humidity_Mean_pct']
    wind = row['Wind_Speed_max_kmh']

    if temp > 35 and humidity > 60: return 2
    elif temp > 40 or temp < 0: return 2
    elif (30 < temp <= 35) or (0 <= temp < 10):
        if wind > 30: return 2
        return 1
    elif 15 <= temp <= 25 and humidity < 70: return 0
    else: return 1

df['Health_Risk_Label'] = df.apply(calculate_health_risk, axis=1)

In [9]:
# 4. Injecting Sensor Noise (Generalization Power-up)
np.random.seed(42)
df['Temp_Mean_C'] += np.random.normal(0, 0.5, len(df))
df['Humidity_Mean_pct'] += np.random.normal(0, 2.0, len(df))

In [10]:
df = df.drop(columns = ['City_Type', 'Date','Temp_Max_C'])

In [11]:
df["Health_Risk_Label"].value_counts()

,count
Health_Risk_Label,
1,32973
0,5763
2,5076


In [12]:
# 5. Feature Selection
features = ['Temp_Mean_C', 'Temp_Fluctuation', 'Humidity_Mean_pct',
            'Heat_Humidity_Index', 'Wind_Speed_max_kmh', 'Is_Raining', 'Precipitation_mm']

X = df[features]
y = df['Health_Risk_Label']

# 6. Training with Global Weights
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
weights = compute_sample_weight(class_weight='balanced', y=y_train)

In [13]:
# XGBoost Configuration for Master Data

xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    max_depth=4,            # Depth control to prevent overfitting
    learning_rate=0.03,     # Slower learning for 43k rows
    reg_lambda=2.0,         # Higher Regularization for better generalization
    n_estimators=200,       # More trees for more data
    random_state=42)

print("Training the Master Brain...")
xgb_model.fit(X_train, y_train, sample_weight=weights)

# 7. Evaluation
print("\n Master Model Validation Report:")
y_pred = xgb_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Safe', 'Moderate', 'High Risk']))

Training the Master Brain...

 Master Model Validation Report:
              precision    recall  f1-score   support

        Safe       0.83      0.99      0.90      1153
    Moderate       1.00      0.96      0.98      6595
   High Risk       0.97      0.99      0.98      1015

    accuracy                           0.97      8763
   macro avg       0.93      0.98      0.95      8763
weighted avg       0.97      0.97      0.97      8763



In [ ]:
filename = 'weather_master_v1.pkl'
with open(filename, 'wb') as file:
    pickle.dump(xgb_model, file)
print(f"Master Model saved as '{filename}'")

Master Model saved as 'weather_master_v3.pkl'


In [15]:
filename = 'weather_master_v3.pkl'
print(f"🧠 Loading Master Brain: {filename}...")
with open(filename, 'rb') as file:
    model = pickle.load(file)

feature_columns = [
    'Temp_Mean_C',
    'Temp_Fluctuation',
    'Humidity_Mean_pct',
    'Heat_Humidity_Index',
    'Wind_Speed_max_kmh',
    'Is_Raining',
    'Precipitation_mm'
]

# 3. Diverse Test Cases (Added 7th value for Precipitation)
test_cases = {
    "Perfect Spring Day": [22.0, 6.0, 40.0, 8.8, 10.0, 0, 0.0],
    "Normal Monsoon Day": [28.0, 4.0, 85.0, 23.8, 12.0, 1, 5.5],
    "Extreme Delhi Heatwave": [41.0, 5.0, 55.0, 22.55, 10.0, 0, 0.0],
    "Amazon Tropical Storm": [33.0, 2.0, 95.0, 31.35, 20.0, 1, 25.0], # Heavy Rain
    "Siberian Winter": [-15.0, 5.0, 60.0, -9.0, 35.0, 0, 0.2],
    "Sahara Peak Summer": [45.0, 8.0, 15.0, 6.75, 15.0, 0, 0.0],
    "Super Cyclone": [34.0, 4.0, 90.0, 30.6, 65.0, 1, 100.0]
}

# 4. Predict
print("\n🔍 Running Master Inference Tests (V3 - 7 Features)...\n")
print(f"{'Scenario':<25} | {'Prediction'}")
print("-" * 45)

for name, features in test_cases.items():
    # Model ko input hamesha DataFrame format mein dena best rehta hai
    df_single = pd.DataFrame([features], columns=feature_columns)
    pred = model.predict(df_single)[0]

    # Label mapping
    label = {0: "Safe", 1: "Moderate", 2: "High Risk"}[pred]
    print(f"{name:<25} | {label}")

print("-" * 45)

🧠 Loading Master Brain: weather_master_v3.pkl...

🔍 Running Master Inference Tests (V3 - 7 Features)...

Scenario                  | Prediction
---------------------------------------------
Perfect Spring Day        | Safe
Normal Monsoon Day        | Moderate
Extreme Delhi Heatwave    | Moderate
Amazon Tropical Storm     | Moderate
Siberian Winter           | High Risk
Sahara Peak Summer        | Moderate
Super Cyclone             | High Risk
---------------------------------------------
